# 02b -- Stage 2 (part 2): Train on Already-Extracted DFDC Frames

**Why this notebook exists**: extraction (~6 hours on CPU for the full
10-part set) plus training (which needs a GPU to be remotely fast -- CPU
training measured at ~2.8 hours *per epoch* for this dataset size) don't
fit together in a single Kaggle session. Running the whole thing as one
unattended commit either exceeds the session time limit (killing the run
with nothing saved) or wastes your GPU-hour quota sitting idle through the
entire CPU-bound extraction phase.

**The fix**: split them.
1. Run `02_stage2_dfdc_frame_extraction_and_train.ipynb`'s extraction cells
   ONLY (skip/remove the training cells for that run), accelerator set to
   **CPU ("None")** -- doesn't touch your GPU quota, and fits in one
   session. Finish it as a successful **Save & Run All (Commit)**.
2. From that successful version's Output tab, click **"New Dataset"**
   (or the equivalent "create dataset from output" option) to turn
   `data/dfdc_frame_manifest.csv` + `data/synthetic_data/*.jpg` into a
   reusable Kaggle Dataset.
3. Attach that new dataset to **this** notebook via "Add Input", set the
   accelerator to **GPU**, and run only this notebook -- no re-extraction,
   so 100% of your GPU time goes to actual training.

Update `FRAME_DATASET_SLUG` below to whatever your new dataset's input
folder is named.

In [ ]:
import sys, os

IS_COLAB = os.path.exists("/var/colab/hostname")
IS_KAGGLE = (not IS_COLAB) and (os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ)
PROJECT_ROOT = "/kaggle/working/ForensicAI_Lab" if IS_KAGGLE else "/content/ForensicAI_Lab"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/MRoy03/Forensic_LAB.git {PROJECT_ROOT}

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
elif IS_KAGGLE:
    !pip install -q -r requirements-train.txt
else:
    print("Not in Colab/Kaggle -- assuming requirements-train.txt is already installed locally.")

In [ ]:
import torch, cv2, numpy as np, pandas as pd, os
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

from config.loader import get_settings, resolve_path
from services.preprocessing.pipeline import preprocess_image
from services.detection.model import ForensicMultiHeadNet

settings = get_settings()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)
if device == "cpu":
    print("[WARNING] No GPU detected -- make sure the accelerator is set to GPU "
          "in this notebook's Settings before running the training cell, or this "
          "defeats the entire point of splitting extraction out.")

## Locate and remap the attached frame dataset

The manifest's `filename` column stores **absolute paths from the
original extraction run** (e.g.
`/kaggle/working/ForensicAI_Lab/data/synthetic_data/...`), which won't
exist in this fresh container. This finds wherever the manifest and
frame images actually landed in the attached dataset, and rewrites the
paths to match -- matching by filename alone, which is safe since every
frame filename is already unique (prefixed by part id + video name +
frame index).

In [ ]:
FRAME_DATASET_SLUG = "dfdc-extracted-frames"  # <-- change to your actual dataset's input folder name
frame_dataset_root = Path("/kaggle/input") / FRAME_DATASET_SLUG
print("Looking under:", frame_dataset_root)


def find_files(root: Path, suffix: str) -> list[Path]:
    matches = []
    for dirpath, _dirnames, filenames in os.walk(root, followlinks=True):
        for f in filenames:
            if f.lower().endswith(suffix):
                matches.append(Path(dirpath) / f)
    return matches


manifest_candidates = find_files(frame_dataset_root, "dfdc_frame_manifest.csv")
assert manifest_candidates, f"No dfdc_frame_manifest.csv found under {frame_dataset_root} -- check FRAME_DATASET_SLUG"
old_manifest_path = manifest_candidates[0]
print("Found manifest at:", old_manifest_path)

jpg_candidates = find_files(frame_dataset_root, ".jpg")
assert jpg_candidates, f"No .jpg frames found under {frame_dataset_root}"
actual_frame_dir = jpg_candidates[0].parent
print(f"Found {len(jpg_candidates)} frame images under:", actual_frame_dir)

In [ ]:
df = pd.read_csv(old_manifest_path)
df["filename"] = df["filename"].apply(lambda p: str(actual_frame_dir / Path(p).name))

missing = (~df["filename"].apply(lambda p: Path(p).exists())).sum()
if missing:
    print(f"[WARNING] {missing} of {len(df)} remapped paths don't actually exist -- "
          "something about the dataset's folder structure may differ from what's assumed here.")

frame_manifest_path = resolve_path("data/dfdc_frame_manifest.csv")
frame_manifest_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(frame_manifest_path, index=False)
print(f"Remapped {len(df)} rows -> {frame_manifest_path}")

## Train (with a held-out validation split)

Training loss alone can't tell you whether the model is learning
transferable "what makes this fake" signal versus just memorizing
specifics of the training frames -- especially relevant for DFDC, where
frames from the same source video/part can share near-identical
faces/backgrounds. `VAL_FRACTION` holds out a slice of frames the model
never trains on; its loss each epoch is the real signal for whether more
epochs are helping (still dropping) or overfitting (flat/rising while
training loss keeps dropping).

In [ ]:
class FrameDataset(Dataset):
    """augment=True applies a random horizontal flip + mild brightness/
    contrast jitter. Needed specifically because the minority (real)
    class gets oversampled ~7x/epoch by the sampler below -- without
    augmentation those repeats are pixel-identical, which the model
    just memorizes instead of learning general real/fake signal (this
    is exactly what caused real_acc to collapse 39%->6% over 5 epochs
    in practice). Set False for validation -- no augmentation there,
    so the reported metrics stay an honest read on genuinely-unseen data.
    """

    def __init__(self, manifest_csv, image_size, augment=False):
        self.df = pd.read_csv(manifest_csv)
        self.image_size = image_size
        self.augment = augment
        self.label_to_idx = {l: i for i, l in enumerate(settings.labels.binary)}
        manip_classes = settings.labels.manipulation_types
        self.manip_to_idx = {m: i for i, m in enumerate(manip_classes)}
        self.none_type = settings.labels.manipulation_type_none

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_bgr = cv2.imread(row["filename"])
        if image_bgr is None:
            print(f"[WARN] Could not read {row['filename']}, substituting another sample")
            return self[(idx + 1) % len(self)]

        if self.augment:
            if random.random() < 0.5:
                image_bgr = cv2.flip(image_bgr, 1)
            alpha = random.uniform(0.85, 1.15)  # contrast
            beta = random.uniform(-20, 20)  # brightness
            image_bgr = cv2.convertScaleAbs(image_bgr, alpha=alpha, beta=beta)

        x = preprocess_image(image_bgr, tuple(self.image_size))

        label_idx = self.label_to_idx[row["label"]]
        row_type = row["type"]
        # "none" (real frames) isn't one of the 6 manipulation_types classes,
        # so .get() would silently default it to class 0 (copy_move) and
        # train the manip head against a wrong label on every real frame --
        # has_manip=0 excludes those from manip_loss instead (same pattern
        # as has_mask).
        has_manip = 0.0 if row_type == self.none_type else 1.0
        manip_idx = self.manip_to_idx.get(row_type, 0)
        mask = np.zeros((1, *self.image_size), dtype=np.float32)

        return (
            torch.from_numpy(x),
            torch.tensor(label_idx, dtype=torch.long),
            torch.tensor(manip_idx, dtype=torch.long),
            torch.from_numpy(mask),
            torch.tensor(0.0, dtype=torch.float32),
            torch.tensor(has_manip, dtype=torch.float32),
        )

In [ ]:
import random

VAL_FRACTION = 0.15

# Split by index first (not torch.utils.data.random_split on one shared
# Dataset instance), so train and val can use separately-configured
# FrameDataset instances -- augmentation must never leak into validation.
manifest_length = len(pd.read_csv(frame_manifest_path))
val_size = int(manifest_length * VAL_FRACTION)
train_size = manifest_length - val_size
generator = torch.Generator().manual_seed(42)
perm = torch.randperm(manifest_length, generator=generator).tolist()
train_indices, val_indices = perm[:train_size], perm[train_size:]

train_dataset_full = FrameDataset(frame_manifest_path, settings.inference.image_size, augment=True)
val_dataset_full = FrameDataset(frame_manifest_path, settings.inference.image_size, augment=False)
train_dataset = torch.utils.data.Subset(train_dataset_full, train_indices)
val_dataset = torch.utils.data.Subset(val_dataset_full, val_indices)

# Oversample the minority class so each batch sees a more balanced mix,
# but with sqrt-of-inverse-frequency weights rather than full inverse
# frequency -- full inverse frequency forces an exact 50/50 split, which
# at this severity (~93%/7%) means repeating each of the ~3,000 unique
# real images ~7x/epoch; sqrt weighting is a milder, still-helpful
# rebalance that repeats the minority class less aggressively.
train_labels = train_dataset_full.df.iloc[train_indices]["label"]
train_label_counts = train_labels.value_counts()
sample_weights = train_labels.map(lambda l: 1.0 / (train_label_counts[l] ** 0.5)).values
sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
print(f"Train: {len(train_dataset)} frames, Val: {len(val_dataset)} frames")

**Check the label balance before trusting accuracy numbers.** DFDC is
known to be heavily imbalanced (official releases run roughly
80-85% fake / 15-20% real) -- if that holds here, a trivial
"always predict fake" model would already score ~85-90% without
learning anything, making a raw accuracy number alone uninterpretable.
That's why the training cell below oversamples the minority class
(`WeightedRandomSampler`) and reports per-class + balanced accuracy,
not just overall accuracy.

In [ ]:
val_df = val_dataset_full.df.iloc[val_indices]
print("Validation label distribution:")
print(val_df["label"].value_counts(normalize=True).round(3))

In [ ]:
# pretrained=True loads ImageNet weights into the encoder (timm adapts
# the first conv layer to in_chans=5 automatically). Every run before
# this was trained from random init -- none of these notebooks actually
# passed `pretrained=` through, so it silently used the class default
# (False) regardless of this setting. A from-scratch CNN needs far more
# data than these datasets provide; this is likely the dominant reason
# results stayed near chance even after fixing sampling/loss weighting
# and adding face-cropping.
model = ForensicMultiHeadNet(
    backbone=settings.models.backbone,
    in_chans=settings.inference.input_channels,
    num_manip_classes=len(settings.labels.manipulation_types),
    pretrained=settings.models.pretrained_encoder,
).to(device)

image_ckpt = resolve_path(settings.models.image_model_checkpoint)
if image_ckpt.exists():
    model.load_state_dict(torch.load(image_ckpt, map_location=device))
    print("Warm-started from Stage 1 checkpoint:", image_ckpt)
else:
    print("No Stage 1 checkpoint found -- training from randomly-initialized weights.")

# weight_decay adds mild L2 regularization -- a cheap first lever
# against overfitting (train_loss dropping while val_loss rises).
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
cls_loss_fn = torch.nn.CrossEntropyLoss()
manip_loss_fn = torch.nn.CrossEntropyLoss(reduction="none")
mask_loss_fn = torch.nn.BCEWithLogitsLoss(reduction="none")

NUM_EPOCHS = 5
checkpoint_path = resolve_path(settings.models.video_model_checkpoint)
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def compute_loss(out, label_idx, manip_idx, mask, has_mask, has_manip):
    loss_cls = cls_loss_fn(out["cls_logits"], label_idx)
    manip_loss_per_sample = manip_loss_fn(out["manip_logits"], manip_idx)
    loss_manip = (manip_loss_per_sample * has_manip).sum() / (has_manip.sum() + 1e-8)
    mask_loss_per_px = mask_loss_fn(out["mask_logits"], mask)
    loss_mask = (mask_loss_per_px.mean(dim=[1, 2, 3]) * has_mask).sum() / (has_mask.sum() + 1e-8)
    return loss_cls + loss_manip + loss_mask


@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    real_idx = settings.labels.binary.index("real")
    real_correct = real_total = fake_correct = fake_total = 0

    for x, label_idx, manip_idx, mask, has_mask, has_manip in loader:
        x, label_idx, manip_idx = x.to(device), label_idx.to(device), manip_idx.to(device)
        mask, has_mask, has_manip = mask.to(device), has_mask.to(device), has_manip.to(device)
        out = model(x)
        total_loss += compute_loss(out, label_idx, manip_idx, mask, has_mask, has_manip).item()

        preds = out["cls_logits"].argmax(dim=1)
        correct_mask = preds == label_idx
        correct += correct_mask.sum().item()
        total += label_idx.size(0)

        is_real = label_idx == real_idx
        real_total += is_real.sum().item()
        real_correct += (correct_mask & is_real).sum().item()
        fake_total += (~is_real).sum().item()
        fake_correct += (correct_mask & ~is_real).sum().item()

    model.train()
    real_acc = real_correct / real_total if real_total else float("nan")
    fake_acc = fake_correct / fake_total if fake_total else float("nan")
    balanced_acc = (real_acc + fake_acc) / 2
    return total_loss / len(loader), correct / total, real_acc, fake_acc, balanced_acc


# Tracks the best validation loss seen and saves a checkpoint only when
# it improves -- the final epoch is NOT necessarily the best one (an
# epoch can have the lowest train_loss while val_loss has already
# started rising, i.e. overfitting), so saving unconditionally at the
# end would keep the wrong weights.
best_val_loss = float("inf")

model.train()
for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    for x, label_idx, manip_idx, mask, has_mask, has_manip in train_loader:
        x, label_idx, manip_idx = x.to(device), label_idx.to(device), manip_idx.to(device)
        mask, has_mask, has_manip = mask.to(device), has_mask.to(device), has_manip.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = compute_loss(out, label_idx, manip_idx, mask, has_mask, has_manip)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    val_loss, val_acc, real_acc, fake_acc, balanced_acc = evaluate(val_loader)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} -- train_loss: {train_loss:.4f}  val_loss: {val_loss:.4f}  "
          f"val_acc: {val_acc:.2%}  real_acc: {real_acc:.2%}  fake_acc: {fake_acc:.2%}  "
          f"balanced_acc: {balanced_acc:.2%}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), checkpoint_path)
        print(f"  -> new best val_loss ({val_loss:.4f}), saved checkpoint to {checkpoint_path}")

print(f"\nBest val_loss achieved: {best_val_loss:.4f} -- that epoch's weights are what's saved, "
      "not necessarily the final epoch's.")

## Getting the checkpoint out of Kaggle

Download `models/video_model.pth` from this run's Output tab and upload
it to `ForensicAI_Lab/models/video_model.pth` on your Drive, same as
before. This is the **best-validation-loss** checkpoint, not necessarily
from the last epoch.